In [1]:
# %pip install seaborn

In [2]:
"""
Unified Stock Price Movement Classification Model - All 10 Stocks
==================================================================
Predicts next-day price movement direction (DOWN/FLAT/UP) for all stocks
Baseline Models: Logistic Regression, XGBoost, Random Forest, SVM
Includes Hyperparameter Tuning with GridSearchCV
Uses pre-split train_data.csv, val_data.csv, and test_data.csv files
Stock encoding included as feature for model differentiation
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, cross_val_score, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.utils import class_weight
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix,
    f1_score,
    make_scorer,
    balanced_accuracy_score
)
import xgboost as xgb
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("="*80)
print("UNIFIED STOCK PRICE MOVEMENT CLASSIFICATION - ALL 10 STOCKS")
print("Using Pre-Split Data with HYPERPARAMETER TUNING")
print("="*80)
print()

UNIFIED STOCK PRICE MOVEMENT CLASSIFICATION - ALL 10 STOCKS
Using Pre-Split Data with HYPERPARAMETER TUNING



In [3]:
# ============================================================================
# 1. DATA LOADING
# ============================================================================
print("Step 1: Loading Data for All Stocks")
print("-" * 80)

# Load datasets from current directory
train_df = pd.read_csv('train_data.csv')
val_df = pd.read_csv('val_data.csv')
test_df = pd.read_csv('test_data.csv')

# Convert date to datetime and sort
train_df['date'] = pd.to_datetime(train_df['date'])
val_df['date'] = pd.to_datetime(val_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

train_df = train_df.sort_values(['stock', 'date']).reset_index(drop=True)
val_df = val_df.sort_values(['stock', 'date']).reset_index(drop=True)
test_df = test_df.sort_values(['stock', 'date']).reset_index(drop=True)

print(f"Training data shape: {train_df.shape}")
print(f"Validation data shape: {val_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"\nTraining date range: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"Validation date range: {val_df['date'].min()} to {val_df['date'].max()}")
print(f"Test date range: {test_df['date'].min()} to {test_df['date'].max()}")
print()

# Display stock distribution
print("Stock Distribution in Training Data:")
stock_dist = train_df['stock'].value_counts().sort_index()
print(stock_dist)
print()

# Check class distribution
print("Class Distribution in Training Data:")
class_dist = train_df['target'].value_counts().sort_index()
print(class_dist)
print(f"\nClass proportions:")
print((class_dist / len(train_df)).round(3))
print()

# Class distribution per stock
print("Class Distribution by Stock:")
print(train_df.groupby(['stock', 'target']).size().unstack(fill_value=0))
print()

Step 1: Loading Data for All Stocks
--------------------------------------------------------------------------------
Training data shape: (6184, 23)
Validation data shape: (1552, 23)
Test data shape: (447, 23)

Training date range: 2023-01-03 00:00:00 to 2025-06-26 00:00:00
Validation date range: 2025-05-22 00:00:00 to 2025-12-26 00:00:00
Test date range: 2026-01-05 00:00:00 to 2026-02-27 00:00:00

Stock Distribution in Training Data:
stock
AP       605
AREIT    618
CNVRG    610
DMC      624
JFC      638
MBT      594
SCC      596
SM       611
SMPH     648
TEL      640
Name: count, dtype: int64

Class Distribution in Training Data:
target
DOWN    1398
FLAT    3392
UP      1394
Name: count, dtype: int64

Class proportions:
target
DOWN    0.226
FLAT    0.549
UP      0.225
Name: count, dtype: float64

Class Distribution by Stock:
target  DOWN  FLAT   UP
stock                  
AP       111   383  111
AREIT     90   421  107
CNVRG    189   232  189
DMC      131   359  134
JFC      159   325

In [4]:
# ============================================================================
# 2. FEATURE ENGINEERING & PREPROCESSING
# ============================================================================
print("Step 2: Feature Engineering & Preprocessing")
print("-" * 80)

# One-hot encode the 'stock' column
print("One-hot encoding 'stock' column...")
train_encoded = pd.get_dummies(train_df, columns=['stock'], prefix='stock', drop_first=False)
val_encoded = pd.get_dummies(val_df, columns=['stock'], prefix='stock', drop_first=False)
test_encoded = pd.get_dummies(test_df, columns=['stock'], prefix='stock', drop_first=False)

# Ensure validation and test sets have all stock columns
train_stock_cols = [col for col in train_encoded.columns if col.startswith('stock_')]
val_stock_cols = [col for col in val_encoded.columns if col.startswith('stock_')]
test_stock_cols = [col for col in test_encoded.columns if col.startswith('stock_')]

for col in train_stock_cols:
    if col not in val_stock_cols:
        val_encoded[col] = 0
    if col not in test_stock_cols:
        test_encoded[col] = 0

# Align column order
val_encoded = val_encoded[train_encoded.columns]
test_encoded = test_encoded[train_encoded.columns]

print(f"Stock columns created: {len(train_stock_cols)}")
print(f"Stock tickers: {sorted([col.replace('stock_', '') for col in train_stock_cols])}")
print()

# Define columns to drop
cols_to_drop = ['date', 'next_close', 'price_change_pct', 'target']

# Separate features and target for all three sets
X_train = train_encoded.drop(cols_to_drop + ['target_encoded'], axis=1)
y_train = train_encoded['target_encoded']
dates_train = train_df['date']

X_val = val_encoded.drop(cols_to_drop + ['target_encoded'], axis=1)
y_val = val_encoded['target_encoded']
dates_val = val_df['date']

X_test = test_encoded.drop(cols_to_drop + ['target_encoded'], axis=1)
y_test = test_encoded['target_encoded']
dates_test = test_df['date']

print(f"Feature matrix shape (training): {X_train.shape}")
print(f"Feature matrix shape (validation): {X_val.shape}")
print(f"Feature matrix shape (test): {X_test.shape}")
print(f"Number of features: {X_train.shape[1]}")
print()

# Calculate balanced class weights
print("\nCalculating balanced class weights...")
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print(f"Class weights:")
for class_idx, weight in class_weight_dict.items():
    print(f"  Class {class_idx}: {weight:.4f}")

# Compute sample weights for XGBoost (map each sample to its class weight)
# sample_weights = np.array([class_weight_dict[y] for y in y_train])
# print(f"Sample weights computed: {len(sample_weights)} weights")

Step 2: Feature Engineering & Preprocessing
--------------------------------------------------------------------------------
One-hot encoding 'stock' column...
Stock columns created: 10
Stock tickers: ['AP', 'AREIT', 'CNVRG', 'DMC', 'JFC', 'MBT', 'SCC', 'SM', 'SMPH', 'TEL']

Feature matrix shape (training): (6184, 27)
Feature matrix shape (validation): (1552, 27)
Feature matrix shape (test): (447, 27)
Number of features: 27


Calculating balanced class weights...
Class weights:
  Class 0: 1.4745
  Class 1: 0.6077
  Class 2: 1.4787


In [5]:
# ============================================================================
# 3. DATA SPLIT SUMMARY
# ============================================================================
print("Step 3: Data Split Summary")
print("-" * 80)

print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")
print()

print(f"Training date range: {dates_train.min()} to {dates_train.max()}")
print(f"Validation date range: {dates_val.min()} to {dates_val.max()}")
print(f"Test date range: {dates_test.min()} to {dates_test.max()}")
print()

# Check class distribution
print("Class distribution across datasets:")
train_class_dist = y_train.value_counts().sort_index()
val_class_dist = y_val.value_counts().sort_index()
test_class_dist = y_test.value_counts().sort_index()

print(f"Training: {train_class_dist.to_dict()}")
print(f"  Proportions: {(train_class_dist / len(y_train)).round(3).to_dict()}")
print(f"Validation: {val_class_dist.to_dict()}")
print(f"  Proportions: {(val_class_dist / len(y_val)).round(3).to_dict()}")
print(f"Test: {test_class_dist.to_dict()}")
print(f"  Proportions: {(test_class_dist / len(y_test)).round(3).to_dict()}")
print()

Step 3: Data Split Summary
--------------------------------------------------------------------------------
Training set size: 6184
Validation set size: 1552
Test set size: 447

Training date range: 2023-01-03 00:00:00 to 2025-06-26 00:00:00
Validation date range: 2025-05-22 00:00:00 to 2025-12-26 00:00:00
Test date range: 2026-01-05 00:00:00 to 2026-02-27 00:00:00

Class distribution across datasets:
Training: {0.0: 1398, 1.0: 3392, 2.0: 1394}
  Proportions: {0.0: 0.226, 1.0: 0.549, 2.0: 0.225}
Validation: {0.0: 310, 1.0: 952, 2.0: 290}
  Proportions: {0.0: 0.2, 1.0: 0.613, 2.0: 0.187}
Test: {0.0: 83, 1.0: 269, 2.0: 95}
  Proportions: {0.0: 0.186, 1.0: 0.602, 2.0: 0.213}



In [6]:
# ============================================================================
# 4. FEATURE SCALING
# ============================================================================
print("Step 4: Feature Scaling")
print("-" * 80)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Features scaled using StandardScaler (fitted on training set only)")
print(f"Scaler mean shape: {scaler.mean_.shape}")
print(f"Scaler scale shape: {scaler.scale_.shape}")
print()

Step 4: Feature Scaling
--------------------------------------------------------------------------------
Features scaled using StandardScaler (fitted on training set only)
Scaler mean shape: (27,)
Scaler scale shape: (27,)



In [7]:
# ============================================================================
# 5. HYPERPARAMETER TUNING
# ============================================================================
print("Step 5: Hyperparameter Tuning with GridSearchCV")
print("=" * 80)
print("Note: This may take some time due to the large dataset and parameter space...")
print()

# Time Series Cross-Validation for tuning
tscv = TimeSeriesSplit(n_splits=3)

# Scoring metric
f1_scorer = make_scorer(f1_score, average='weighted')

tuned_models = {}
best_params_dict = {}

# --- 1. Logistic Regression Tuning ---
print("\n[1/4] Tuning Logistic Regression")
print("-" * 80)

lr_param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [1000]
}

lr_grid = GridSearchCV(
    LogisticRegression(class_weight=class_weight_dict, random_state=RANDOM_STATE),
    lr_param_grid,
    cv=tscv,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=2
)

print("Starting grid search...")
lr_grid.fit(X_train_scaled, y_train)
best_params_dict['Logistic Regression'] = lr_grid.best_params_
tuned_models['Logistic Regression'] = lr_grid.best_estimator_

print(f"✓ Best parameters: {lr_grid.best_params_}")
print(f"✓ Best CV score (F1): {lr_grid.best_score_:.4f}")

# --- 2. XGBoost Tuning ---
print("\n[2/4] Tuning XGBoost")
print("-" * 80)

xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_grid = GridSearchCV(
    xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='mlogloss'),
    xgb_param_grid,
    cv=tscv,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=2
)

print("Starting grid search...")
xgb_grid.fit(X_train_scaled, y_train)
best_params_dict['XGBoost'] = xgb_grid.best_params_
tuned_models['XGBoost'] = xgb_grid.best_estimator_

print(f"✓ Best parameters: {xgb_grid.best_params_}")
print(f"✓ Best CV score (F1): {xgb_grid.best_score_:.4f}")

# --- 3. Random Forest Tuning ---
print("\n[3/4] Tuning Random Forest")
print("-" * 80)

rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    rf_param_grid,
    cv=tscv,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=2
)

print("Starting grid search...")
rf_grid.fit(X_train_scaled, y_train)
best_params_dict['Random Forest'] = rf_grid.best_params_
tuned_models['Random Forest'] = rf_grid.best_estimator_

print(f"✓ Best parameters: {rf_grid.best_params_}")
print(f"✓ Best CV score (F1): {rf_grid.best_score_:.4f}")

# --- 4. SVM Tuning ---
print("\n[4/4] Tuning Support Vector Machine (SVM)")
print("-" * 80)

svm_param_grid = {
    'C': [0.1, 1.0, 10.0],
    'kernel': ['rbf', 'poly'],
    'gamma': ['scale', 'auto'],
    'degree': [2, 3]
}

svm_grid = GridSearchCV(
    SVC(class_weight=class_weight_dict, random_state=RANDOM_STATE),
    svm_param_grid,
    cv=tscv,
    scoring=f1_scorer,
    n_jobs=-1,
    verbose=2
)

print("Starting grid search...")
svm_grid.fit(X_train_scaled, y_train)
best_params_dict['SVM'] = svm_grid.best_params_
tuned_models['SVM'] = svm_grid.best_estimator_

print(f"✓ Best parameters: {svm_grid.best_params_}")
print(f"✓ Best CV score (F1): {svm_grid.best_score_:.4f}")

Step 5: Hyperparameter Tuning with GridSearchCV
Note: This may take some time due to the large dataset and parameter space...


[1/4] Tuning Logistic Regression
--------------------------------------------------------------------------------
Starting grid search...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END ....C=0.01, max_iter=1000, penalty=l2, solver=lbfgs; total time=   0.0s
[CV] END ....C=0.01, max_iter=1000, penalty=l2, solver=lbfgs; total time=   0.0s
[CV] END .....C=0.01, max_iter=1000, penalty=l2, solver=saga; total time=   0.0s
[CV] END .....C=0.1, max_iter=1000, penalty=l2, solver=lbfgs; total time=   0.0s
[CV] END ....C=0.01, max_iter=1000, penalty=l2, solver=lbfgs; total time=   0.0s
[CV] END .....C=1.0, max_iter=1000, penalty=l2, solver=lbfgs; total time=   0.0s
[CV] END .....C=0.1, max_iter=1000, penalty=l2, solver=lbfgs; total time=   0.0s
[CV] END .....C=0.1, max_iter=1000, penalty=l2, solver=lbfgs; total time=   0.0s
[CV] END .....C=1.0, max_i

/Users/joshuarosell/Documents/ML TUNING/.venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/joshuarosell/Documents/ML TUNING/.venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END .....C=10.0, max_iter=1000, penalty=l2, solver=saga; total time=   0.5s


/Users/joshuarosell/Documents/ML TUNING/.venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/joshuarosell/Documents/ML TUNING/.venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END ......C=1.0, max_iter=1000, penalty=l2, solver=saga; total time=   0.9s
[CV] END .....C=10.0, max_iter=1000, penalty=l2, solver=saga; total time=   0.9s


/Users/joshuarosell/Documents/ML TUNING/.venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/joshuarosell/Documents/ML TUNING/.venv/lib/python3.9/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] END ......C=1.0, max_iter=1000, penalty=l2, solver=saga; total time=   1.3s
[CV] END .....C=10.0, max_iter=1000, penalty=l2, solver=saga; total time=   1.3s
✓ Best parameters: {'C': 0.01, 'max_iter': 1000, 'penalty': 'l2', 'solver': 'lbfgs'}
✓ Best CV score (F1): 0.3995

[2/4] Tuning XGBoost
--------------------------------------------------------------------------------
Starting grid search...
Fitting 3 folds for each of 108 candidates, totalling 324 fits
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.8; total time=   0.1s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.8; total time=   0.1s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimators=50, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.8, learning_rate=0.01, max_depth=3, n_estimato

In [8]:
# ============================================================================
# 6. MODEL EVALUATION WITH TUNED PARAMETERS
# ============================================================================
print("\n" + "="*80)
print("Step 6: Evaluating Tuned Models")
print("="*80)

results = {
    'model': [],
    'train_accuracy': [],
    'val_accuracy': [],
    'test_accuracy': [],
    'train_balanced_accuracy': [],
    'val_balanced_accuracy': [],
    'test_balanced_accuracy': [],
    'train_f1_weighted': [],
    'val_f1_weighted': [],
    'test_f1_weighted': [],
    'best_params': []
}

predictions = {}

for model_name, model in tuned_models.items():
    print(f"\n{model_name}")
    print("-" * 80)
    
    # Predictions
    train_pred = model.predict(X_train_scaled)
    val_pred = model.predict(X_val_scaled)
    test_pred = model.predict(X_test_scaled)
    
    predictions[model_name] = (train_pred, val_pred, test_pred)
    
    # Metrics
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)
    train_f1 = f1_score(y_train, train_pred, average='weighted')
    val_f1 = f1_score(y_val, val_pred, average='weighted')
    test_f1 = f1_score(y_test, test_pred, average='weighted')
    train_bal_acc = balanced_accuracy_score(y_train, train_pred)
    val_bal_acc = balanced_accuracy_score(y_val, val_pred)
    test_bal_acc = balanced_accuracy_score(y_test, test_pred)
    
    print(f"Train - Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val   - Acc: {val_acc:.4f} | F1: {val_f1:.4f}")
    print(f"Test  - Acc: {test_acc:.4f} | F1: {test_f1:.4f}")

    print(f"Train - Acc: {train_acc:.4f} | Bal Acc: {train_bal_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Val   - Acc: {val_acc:.4f} | Bal Acc: {val_bal_acc:.4f} | F1: {val_f1:.4f}")
    print(f"Test  - Acc: {test_acc:.4f} | Bal Acc: {test_bal_acc:.4f} | F1: {test_f1:.4f}")
    
    # Store all metrics
    results['train_balanced_accuracy'].append(train_bal_acc)
    results['val_balanced_accuracy'].append(val_bal_acc)
    results['test_balanced_accuracy'].append(test_bal_acc)
    results['model'].append(model_name)
    results['train_accuracy'].append(train_acc)
    results['val_accuracy'].append(val_acc)
    results['test_accuracy'].append(test_acc)
    results['train_f1_weighted'].append(train_f1)
    results['val_f1_weighted'].append(val_f1)
    results['test_f1_weighted'].append(test_f1)
    results['best_params'].append(str(best_params_dict[model_name]))


Step 6: Evaluating Tuned Models

Logistic Regression
--------------------------------------------------------------------------------
Train - Acc: 0.4505 | F1: 0.4630
Val   - Acc: 0.4923 | F1: 0.5050
Test  - Acc: 0.5593 | F1: 0.5571
Train - Acc: 0.4505 | Bal Acc: 0.4131 | F1: 0.4630
Val   - Acc: 0.4923 | Bal Acc: 0.4011 | F1: 0.5050
Test  - Acc: 0.5593 | Bal Acc: 0.4523 | F1: 0.5571

XGBoost
--------------------------------------------------------------------------------
Train - Acc: 0.5817 | F1: 0.4751
Val   - Acc: 0.6031 | F1: 0.4985
Test  - Acc: 0.6152 | F1: 0.5017
Train - Acc: 0.5817 | Bal Acc: 0.3920 | F1: 0.4751
Val   - Acc: 0.6031 | Bal Acc: 0.3529 | F1: 0.4985
Test  - Acc: 0.6152 | Bal Acc: 0.3662 | F1: 0.5017

Random Forest
--------------------------------------------------------------------------------
Train - Acc: 0.5558 | F1: 0.4108
Val   - Acc: 0.6134 | F1: 0.4718
Test  - Acc: 0.6040 | F1: 0.4578
Train - Acc: 0.5558 | Bal Acc: 0.3470 | F1: 0.4108
Val   - Acc: 0.6134 | Bal

In [9]:
# ============================================================================
# 7. TIME SERIES CROSS-VALIDATION ON TRAINING SET
# ============================================================================
print("\n" + "="*80)
print("Step 7: Time Series Cross-Validation (5 Splits)")
print("="*80)

cv_strategy = TimeSeriesSplit(n_splits=5)
cv_results = {}

for model_name, model in tuned_models.items():
    print(f"\n{model_name}:")
    cv_accuracy = cross_val_score(model, X_train_scaled, y_train, 
                                   cv=cv_strategy, scoring='accuracy', n_jobs=-1)
    cv_f1 = cross_val_score(model, X_train_scaled, y_train, 
                            cv=cv_strategy, scoring='f1_weighted', n_jobs=-1)
    
    cv_results[model_name] = {
        'accuracy': cv_accuracy,
        'f1': cv_f1
    }
    
    print(f"  CV Accuracy: {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std():.4f})")
    print(f"  CV F1: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})")
    print(f"  Fold scores: {[f'{score:.4f}' for score in cv_accuracy]}")


Step 7: Time Series Cross-Validation (5 Splits)

Logistic Regression:
  CV Accuracy: 0.3957 (+/- 0.0450)
  CV F1: 0.3700 (+/- 0.0348)
  Fold scores: ['0.4534', '0.4398', '0.3922', '0.3505', '0.3427']

XGBoost:
  CV Accuracy: 0.4573 (+/- 0.0639)
  CV F1: 0.3958 (+/- 0.0345)
  Fold scores: ['0.3864', '0.3913', '0.5553', '0.4583', '0.4951']

Random Forest:
  CV Accuracy: 0.4695 (+/- 0.0816)
  CV F1: 0.3690 (+/- 0.0568)
  Fold scores: ['0.3602', '0.3854', '0.5563', '0.5456', '0.5000']

SVM:
  CV Accuracy: 0.4082 (+/- 0.0688)
  CV F1: 0.3844 (+/- 0.0417)
  Fold scores: ['0.4573', '0.3641', '0.5184', '0.3350', '0.3660']


In [10]:
# ============================================================================
# 8. DETAILED EVALUATION ON TEST SET
# ============================================================================
print("\n" + "="*80)
print("Step 8: Detailed Evaluation on Test Set (All Stocks)")
print("="*80)

class_names = ['DOWN (0)', 'FLAT (1)', 'UP (2)']

for model_name in ['Logistic Regression', 'XGBoost', 'Random Forest', 'SVM']:
    print(f"\n{model_name}")
    print("-" * 80)
    
    test_pred = predictions[model_name][2]
    
    print("\nClassification Report:")
    print(classification_report(y_test, test_pred, target_names=class_names, zero_division=0))
    
    cm = confusion_matrix(y_test, test_pred)
    print("\nConfusion Matrix:")
    print(cm)


Step 8: Detailed Evaluation on Test Set (All Stocks)

Logistic Regression
--------------------------------------------------------------------------------

Classification Report:
              precision    recall  f1-score   support

    DOWN (0)       0.28      0.31      0.30        83
    FLAT (1)       0.70      0.72      0.71       269
      UP (2)       0.39      0.33      0.36        95

    accuracy                           0.56       447
   macro avg       0.46      0.45      0.45       447
weighted avg       0.56      0.56      0.56       447


Confusion Matrix:
[[ 26  38  19]
 [ 47 193  29]
 [ 19  45  31]]

XGBoost
--------------------------------------------------------------------------------

Classification Report:
              precision    recall  f1-score   support

    DOWN (0)       0.20      0.01      0.02        83
    FLAT (1)       0.63      0.98      0.77       269
      UP (2)       0.45      0.11      0.17        95

    accuracy                           0.6

In [11]:
# ============================================================================
# 9. PER-STOCK PERFORMANCE ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("Step 9: Per-Stock Performance Analysis")
print("="*80)

# Add stock column back for analysis
test_df_copy = test_df.copy()
test_df_copy['actual'] = y_test.values

per_stock_results = []

for model_name in ['Logistic Regression', 'XGBoost', 'Random Forest', 'SVM']:
    test_df_copy[f'pred_{model_name}'] = predictions[model_name][2]
    
    for stock in sorted(test_df_copy['stock'].unique()):
        stock_mask = test_df_copy['stock'] == stock
        y_stock_true = test_df_copy.loc[stock_mask, 'actual']
        y_stock_pred = test_df_copy.loc[stock_mask, f'pred_{model_name}']
        
        if len(y_stock_true) > 0:
            acc = accuracy_score(y_stock_true, y_stock_pred)
            f1 = f1_score(y_stock_true, y_stock_pred, average='weighted', zero_division=0)
            
            per_stock_results.append({
                'model': model_name,
                'stock': stock,
                'accuracy': acc,
                'f1_weighted': f1,
                'n_samples': len(y_stock_true)
            })

per_stock_df = pd.DataFrame(per_stock_results)

print("\nPer-Stock Test Accuracy:")
pivot_acc = per_stock_df.pivot(index='stock', columns='model', values='accuracy')
print(pivot_acc.to_string())

print("\nPer-Stock Test F1 Score:")
pivot_f1 = per_stock_df.pivot(index='stock', columns='model', values='f1_weighted')
print(pivot_f1.to_string())

print("\nSample counts per stock:")
print(per_stock_df.groupby('stock')['n_samples'].first().to_string())


Step 9: Per-Stock Performance Analysis

Per-Stock Test Accuracy:
model  Logistic Regression  Random Forest       SVM   XGBoost
stock                                                        
AP                0.780488       0.804878  0.780488  0.804878
AREIT             0.789474       0.789474  0.789474  0.789474
CNVRG             0.410256       0.307692  0.307692  0.333333
DMC               0.410256       0.435897  0.487179  0.487179
JFC               0.616667       0.650000  0.316667  0.650000
MBT               0.404762       0.500000  0.452381  0.523810
SCC               0.525000       0.500000  0.475000  0.500000
SM                0.630435       0.695652  0.695652  0.695652
SMPH              0.517857       0.625000  0.607143  0.642857
TEL               0.500000       0.673913  0.478261  0.673913

Per-Stock Test F1 Score:
model  Logistic Regression  Random Forest       SVM   XGBoost
stock                                                        
AP                0.705647       0.71786

In [12]:
# ============================================================================
# 10. VISUALIZE CONFUSION MATRICES
# ============================================================================
print("\n" + "="*80)
print("Step 10: Creating Confusion Matrix Visualizations")
print("="*80)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for idx, model_name in enumerate(['Logistic Regression', 'XGBoost', 'Random Forest', 'SVM']):
    val_pred = predictions[model_name][1]
    test_pred = predictions[model_name][2]
    
    # Validation confusion matrix
    cm_val = confusion_matrix(y_val, val_pred)
    sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['DOWN', 'FLAT', 'UP'],
                yticklabels=['DOWN', 'FLAT', 'UP'],
                ax=axes[0, idx], cbar=False)
    axes[0, idx].set_title(f'{model_name}\nValidation Set (All Stocks)', fontsize=10)
    axes[0, idx].set_ylabel('True Label')
    axes[0, idx].set_xlabel('Predicted Label')
    
    # Test confusion matrix
    cm_test = confusion_matrix(y_test, test_pred)
    sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
                xticklabels=['DOWN', 'FLAT', 'UP'],
                yticklabels=['DOWN', 'FLAT', 'UP'],
                ax=axes[1, idx], cbar=False)
    axes[1, idx].set_title(f'{model_name}\nTest Set (All Stocks)', fontsize=10)
    axes[1, idx].set_ylabel('True Label')
    axes[1, idx].set_xlabel('Predicted Label')

plt.tight_layout()
output_dir = 'models_unified'
os.makedirs(output_dir, exist_ok=True)
plt.savefig(f'{output_dir}/confusion_matrices_unified.png', dpi=300, bbox_inches='tight')
print(f"✓ Confusion matrices saved as '{output_dir}/confusion_matrices_unified.png'")
plt.close()


Step 10: Creating Confusion Matrix Visualizations
✓ Confusion matrices saved as 'models_unified/confusion_matrices_unified.png'


In [13]:
# ============================================================================
# 11. RESULTS SUMMARY
# ============================================================================
print("\n" + "="*80)
print("Step 11: Model Comparison Summary")
print("="*80)

results_df = pd.DataFrame(results)
print("\nOverall Performance Comparison (All 10 Stocks):")
print(results_df[['model', 'train_accuracy', 'val_accuracy', 'test_accuracy', 
                  'train_f1_weighted', 'val_f1_weighted', 'test_f1_weighted']].to_string(index=False))

print("\n\nBest Hyperparameters Found:")
print("-" * 80)
for model_name, params in best_params_dict.items():
    print(f"\n{model_name}:")
    for param, value in params.items():
        print(f"  {param}: {value}")

best_idx = results_df['test_accuracy'].idxmax()
best_model_name = results_df.loc[best_idx, 'model']
best_test_acc = results_df.loc[best_idx, 'test_accuracy']
best_test_f1 = results_df.loc[best_idx, 'test_f1_weighted']

print(f"\n🏆 Best Model: {best_model_name}")
print(f"   Test Accuracy: {best_test_acc:.4f}")
print(f"   Test F1: {best_test_f1:.4f}")


Step 11: Model Comparison Summary

Overall Performance Comparison (All 10 Stocks):
              model  train_accuracy  val_accuracy  test_accuracy  train_f1_weighted  val_f1_weighted  test_f1_weighted
Logistic Regression        0.450517      0.492268       0.559284           0.462997         0.505011          0.557124
            XGBoost        0.581662      0.603093       0.615213           0.475097         0.498500          0.501718
      Random Forest        0.555789      0.613402       0.604027           0.410849         0.471751          0.457831
                SVM        0.494340      0.473582       0.532438           0.494500         0.482621          0.528033


Best Hyperparameters Found:
--------------------------------------------------------------------------------

Logistic Regression:
  C: 0.01
  max_iter: 1000
  penalty: l2
  solver: lbfgs

XGBoost:
  colsample_bytree: 1.0
  learning_rate: 0.1
  max_depth: 3
  n_estimators: 100
  subsample: 0.8

Random Forest:
  max_de

In [14]:
# ============================================================================
# 12. SAVE MODELS AND ARTIFACTS
# ============================================================================
print("\n" + "="*80)
print("Step 12: Saving Unified Models and Artifacts")
print("="*80)

# Save tuned models
joblib.dump(tuned_models['Logistic Regression'], f'{output_dir}/logistic_regression_unified_tuned.pkl')
joblib.dump(tuned_models['XGBoost'], f'{output_dir}/xgboost_unified_tuned.pkl')
joblib.dump(tuned_models['Random Forest'], f'{output_dir}/random_forest_unified_tuned.pkl')
joblib.dump(tuned_models['SVM'], f'{output_dir}/svm_unified_tuned.pkl')
joblib.dump(scaler, f'{output_dir}/scaler_unified.pkl')

# Save best parameters
joblib.dump(best_params_dict, f'{output_dir}/best_params_unified.pkl')

# Save feature info
feature_info = {
    'feature_names': X_train.columns.tolist(),
    'stock_columns': train_stock_cols,
    'n_features': X_train.shape[1],
    'split_method': 'pre_split_files',
    'train_date_range': (str(dates_train.min()), str(dates_train.max())),
    'val_date_range': (str(dates_val.min()), str(dates_val.max())),
    'test_date_range': (str(dates_test.min()), str(dates_test.max())),
    'stocks': sorted([col.replace('stock_', '') for col in train_stock_cols]),
    'tuning_method': 'GridSearchCV with TimeSeriesSplit',
    'unified_model': True
}
joblib.dump(feature_info, f'{output_dir}/feature_info_unified.pkl')

# Save results
results_df.to_csv(f'{output_dir}/model_comparison_unified.csv', index=False)
per_stock_df.to_csv(f'{output_dir}/per_stock_performance.csv', index=False)

# Save CV results
cv_results_df = pd.DataFrame([
    {
        'model': model_name,
        'cv_accuracy_mean': cv_results[model_name]['accuracy'].mean(),
        'cv_accuracy_std': cv_results[model_name]['accuracy'].std(),
        'cv_f1_mean': cv_results[model_name]['f1'].mean(),
        'cv_f1_std': cv_results[model_name]['f1'].std()
    }
    for model_name in tuned_models.keys()
])
cv_results_df.to_csv(f'{output_dir}/cross_validation_results.csv', index=False)

print(f"\n✓ All unified models saved in: {output_dir}/")
print(f"\n✓ Model files: *_unified_tuned.pkl")
print(f"✓ Best parameters: best_params_unified.pkl")
print(f"✓ Scaler: scaler_unified.pkl")
print(f"✓ Overall results: model_comparison_unified.csv")
print(f"✓ Per-stock performance: per_stock_performance.csv")
print(f"✓ Cross-validation results: cross_validation_results.csv")

print(f"\n" + "="*80)
print("UNIFIED MODEL TRAINING COMPLETE FOR ALL 10 STOCKS!")
print("="*80)
print(f"\nDataset Summary:")
print(f"  Training: {len(X_train)} samples")
print(f"  Validation: {len(X_val)} samples")
print(f"  Test: {len(X_test)} samples")
print(f"  Total samples: {len(X_train) + len(X_val) + len(X_test)}")
print(f"  Features: {X_train.shape[1]} (including {len(train_stock_cols)} stock encodings)")
print(f"  Stocks: {len(train_stock_cols)}")
print(f"  Classes: 3 (DOWN/FLAT/UP)")
print("="*80)


Step 12: Saving Unified Models and Artifacts

✓ All unified models saved in: models_unified/

✓ Model files: *_unified_tuned.pkl
✓ Best parameters: best_params_unified.pkl
✓ Scaler: scaler_unified.pkl
✓ Overall results: model_comparison_unified.csv
✓ Per-stock performance: per_stock_performance.csv
✓ Cross-validation results: cross_validation_results.csv

UNIFIED MODEL TRAINING COMPLETE FOR ALL 10 STOCKS!

Dataset Summary:
  Training: 6184 samples
  Validation: 1552 samples
  Test: 447 samples
  Total samples: 8183
  Features: 27 (including 10 stock encodings)
  Stocks: 10
  Classes: 3 (DOWN/FLAT/UP)


In [17]:
# ============================================================================
# Save feature names into each model's metadata and re-save pkl files
# ============================================================================
print("Saving feature names into model artifacts...")

feature_names = X_train.columns.tolist()

# Update feature_info with feature names (already exists, but ensure it's saved)
feature_info_updated = {
    'feature_names': feature_names,
    'stock_columns': train_stock_cols,
    'n_features': X_train.shape[1],
    'split_method': 'pre_split_files',
    'train_date_range': (str(dates_train.min()), str(dates_train.max())),
    'val_date_range': (str(dates_val.min()), str(dates_val.max())),
    'test_date_range': (str(dates_test.min()), str(dates_test.max())),
    'stocks': sorted([col.replace('stock_', '') for col in train_stock_cols]),
    'tuning_method': 'GridSearchCV with TimeSeriesSplit',
    'unified_model': True
}

# Wrap each model with its feature names in a dict and re-save
for model_name, model in tuned_models.items():
    model_bundle = {
        'model': model,
        'feature_names': feature_names,
        'best_params': best_params_dict[model_name]
    }
    safe_name = model_name.lower().replace(' ', '_')
    save_path = f'{output_dir}/{safe_name}_unified_tuned_with_features.pkl'
    joblib.dump(model_bundle, save_path)
    print(f"✓ Saved {model_name} bundle: {save_path}")

# Also update the feature_info file
joblib.dump(feature_info_updated, f'{output_dir}/feature_info_unified.pkl')
print(f"✓ Updated feature_info_unified.pkl with {len(feature_names)} feature names")
print(f"\nFeature names ({len(feature_names)} total):")
print(feature_names)

Saving feature names into model artifacts...
✓ Saved Logistic Regression bundle: models_unified/logistic_regression_unified_tuned_with_features.pkl
✓ Saved XGBoost bundle: models_unified/xgboost_unified_tuned_with_features.pkl
✓ Saved Random Forest bundle: models_unified/random_forest_unified_tuned_with_features.pkl
✓ Saved SVM bundle: models_unified/svm_unified_tuned_with_features.pkl
✓ Updated feature_info_unified.pkl with 27 feature names

Feature names (27 total):
['open', 'high', 'low', 'close', 'volume', 'positive_sent', 'negative_sent', 'neutral_sent', 'sent_score', 'has_disclosure', 'sma_20', 'ema_12', 'ema_26', 'macd', 'signal_line', 'macd_histogram', 'rsi', 'stock_AP', 'stock_AREIT', 'stock_CNVRG', 'stock_DMC', 'stock_JFC', 'stock_MBT', 'stock_SCC', 'stock_SM', 'stock_SMPH', 'stock_TEL']


In [15]:
# %pip install shap

In [16]:
# # ============================================================================
# # SHAP ANALYSIS FOR XGBOOST MODEL
# # ============================================================================
# print("="*80)
# print("SHAP ANALYSIS - XGBOOST MODEL EXPLAINABILITY")
# print("="*80)
# print("\nSHAP (SHapley Additive exPlanations) helps understand:")
# print("  - Which features are most important for the model")
# print("  - How each feature affects predictions")
# print("  - Individual prediction explanations")
# print()

# import shap

# # Get the XGBoost model
# xgb_model = tuned_models['XGBoost']
# print(f"Model: {type(xgb_model).__name__}")
# print(f"Best parameters: {best_params_dict['XGBoost']}")
# print()

# # ============================================================================
# # 1. CREATE SHAP EXPLAINER
# # ============================================================================
# print("\n[1/5] Creating SHAP Explainer...")
# print("-" * 80)

# # Use TreeExplainer for XGBoost (fast and exact for tree-based models)
# explainer = shap.TreeExplainer(xgb_model)
# print("✓ TreeExplainer created successfully")

# # Calculate SHAP values for test set (use a subset if dataset is large)
# max_samples_for_shap = 500
# if len(X_test_scaled) > max_samples_for_shap:
#     print(f"Using {max_samples_for_shap} random samples from test set for SHAP analysis")
#     shap_indices = np.random.choice(len(X_test_scaled), max_samples_for_shap, replace=False)
#     X_test_shap = X_test_scaled[shap_indices]
#     y_test_shap = y_test.iloc[shap_indices]
# else:
#     print(f"Using all {len(X_test_scaled)} test samples for SHAP analysis")
#     X_test_shap = X_test_scaled
#     y_test_shap = y_test

# print(f"Computing SHAP values for {len(X_test_shap)} samples...")
# shap_values = explainer(X_test_shap)
# print(f"✓ SHAP values computed: shape = {shap_values.values.shape}")
# print(f"  (samples, features, classes): ({shap_values.values.shape[0]}, {shap_values.values.shape[1]}, {shap_values.values.shape[2]})")
# print()

# # ============================================================================
# # 2. GLOBAL FEATURE IMPORTANCE
# # ============================================================================
# print("\n[2/5] Global Feature Importance")
# print("-" * 80)

# # Calculate mean absolute SHAP values for each feature (across all classes)
# feature_importance = np.abs(shap_values.values).mean(axis=(0, 2))
# feature_names = X_train.columns.tolist()

# # Create feature importance dataframe
# feature_importance_df = pd.DataFrame({
#     'Feature': feature_names,
#     'Importance': feature_importance
# }).sort_values('Importance', ascending=False)

# print("\nTop 15 Most Important Features:")
# print(feature_importance_df.head(15).to_string(index=False))
# print()

# # ============================================================================
# # 3. SHAP SUMMARY PLOT (All Classes)
# # ============================================================================
# print("\n[3/5] SHAP Summary Plots")
# print("-" * 80)

# class_names = ['DOWN', 'FLAT', 'UP']

# # Summary plot for each class
# fig, axes = plt.subplots(1, 3, figsize=(24, 8))
# for i, class_name in enumerate(class_names):
#     plt.sca(axes[i])
#     shap.summary_plot(
#         shap_values.values[:, :, i],
#         X_test_shap,
#         feature_names=feature_names,
#         show=False,
#         max_display=15
#     )
#     axes[i].set_title(f'SHAP Summary - Class {i} ({class_name})', fontsize=14, fontweight='bold')

# plt.tight_layout()
# plt.savefig(f'{output_dir}/shap_summary_all_classes.png', dpi=150, bbox_inches='tight')
# print(f"✓ Saved: {output_dir}/shap_summary_all_classes.png")
# plt.show()

# # ============================================================================
# # 4. SHAP BAR PLOT - MEAN FEATURE IMPORTANCE
# # ============================================================================
# print("\n[4/5] SHAP Feature Importance Bar Plot")
# print("-" * 80)

# # Bar plot showing mean absolute SHAP values across all classes
# fig, ax = plt.subplots(figsize=(12, 8))
# shap.summary_plot(
#     shap_values.values.mean(axis=2),  # Average across classes
#     X_test_shap,
#     feature_names=feature_names,
#     plot_type='bar',
#     show=False,
#     max_display=20
# )
# plt.title('Top 20 Features by Mean |SHAP Value| (Averaged Across All Classes)', 
#           fontsize=14, fontweight='bold', pad=20)
# plt.tight_layout()
# plt.savefig(f'{output_dir}/shap_feature_importance_bar.png', dpi=150, bbox_inches='tight')
# print(f"✓ Saved: {output_dir}/shap_feature_importance_bar.png")
# plt.show()

# # ============================================================================
# # 5. SHAP WATERFALL PLOTS - INDIVIDUAL PREDICTIONS
# # ============================================================================
# print("\n[5/5] SHAP Waterfall Plots (Individual Predictions)")
# print("-" * 80)

# # Select a few interesting examples (one from each class if possible)
# sample_indices = []
# for class_idx in range(3):
#     class_mask = y_test_shap == class_idx
#     if class_mask.sum() > 0:
#         class_sample_idx = np.where(class_mask)[0][0]
#         sample_indices.append(class_sample_idx)

# print(f"Generating waterfall plots for {len(sample_indices)} sample predictions...")

# for idx, sample_idx in enumerate(sample_indices):
#     true_class = int(y_test_shap.iloc[sample_idx])
#     pred_class = int(xgb_model.predict(X_test_shap[sample_idx:sample_idx+1])[0])
    
#     # Create waterfall plot for each class
#     fig, axes = plt.subplots(1, 3, figsize=(24, 6))
    
#     for class_idx in range(3):
#         plt.sca(axes[class_idx])
#         shap.waterfall_plot(
#             shap.Explanation(
#                 values=shap_values.values[sample_idx, :, class_idx],
#                 base_values=shap_values.base_values[sample_idx, class_idx],
#                 data=X_test_shap[sample_idx],
#                 feature_names=feature_names
#             ),
#             max_display=15,
#             show=False
#         )
#         axes[class_idx].set_title(
#             f'Class {class_idx} ({class_names[class_idx]})',
#             fontsize=12,
#             fontweight='bold'
#         )
    
#     fig.suptitle(
#         f'SHAP Waterfall - Sample {idx+1} | True: {class_names[true_class]} | Predicted: {class_names[pred_class]}',
#         fontsize=14,
#         fontweight='bold',
#         y=1.02
#     )
#     plt.tight_layout()
#     plt.savefig(f'{output_dir}/shap_waterfall_sample_{idx+1}.png', dpi=150, bbox_inches='tight')
#     print(f"✓ Saved: {output_dir}/shap_waterfall_sample_{idx+1}.png")
#     plt.show()

# # ============================================================================
# # 6. SHAP DEPENDENCE PLOTS - TOP FEATURES
# # ============================================================================
# print("\n[6/6] SHAP Dependence Plots (Top Features)")
# print("-" * 80)

# # Get top 6 features (excluding stock encoding features for clarity)
# top_features = feature_importance_df[~feature_importance_df['Feature'].str.startswith('stock_')].head(6)

# print(f"Creating dependence plots for top 6 non-stock features...")
# print(top_features[['Feature', 'Importance']].to_string(index=False))
# print()

# # Create dependence plots
# fig, axes = plt.subplots(2, 3, figsize=(20, 12))
# axes = axes.flatten()

# for i, (_, row) in enumerate(top_features.iterrows()):
#     feature_name = row['Feature']
#     feature_idx = feature_names.index(feature_name)
    
#     plt.sca(axes[i])
#     # Use mean across classes for dependence plot
#     shap_values_mean = shap_values.values[:, :, :].mean(axis=2)
#     shap.dependence_plot(
#         feature_idx,
#         shap_values_mean,
#         X_test_shap,
#         feature_names=feature_names,
#         show=False,
#         ax=axes[i]
#     )
#     axes[i].set_title(f'Dependence: {feature_name}', fontsize=12, fontweight='bold')

# plt.suptitle('SHAP Dependence Plots - Top 6 Features', fontsize=16, fontweight='bold', y=1.00)
# plt.tight_layout()
# plt.savefig(f'{output_dir}/shap_dependence_plots.png', dpi=150, bbox_inches='tight')
# print(f"✓ Saved: {output_dir}/shap_dependence_plots.png")
# plt.show()

# # ============================================================================
# # SUMMARY
# # ============================================================================
# print("\n" + "="*80)
# print("SHAP ANALYSIS COMPLETE!")
# print("="*80)
# print("\nGenerated visualizations:")
# print(f"  1. {output_dir}/shap_summary_all_classes.png")
# print(f"  2. {output_dir}/shap_feature_importance_bar.png")
# for idx in range(len(sample_indices)):
#     print(f"  {3+idx}. {output_dir}/shap_waterfall_sample_{idx+1}.png")
# print(f"  {3+len(sample_indices)}. {output_dir}/shap_dependence_plots.png")
# print("\nKey Insights:")
# print(f"  • Top feature: {feature_importance_df.iloc[0]['Feature']}")
# print(f"  • Model uses {len(feature_names)} features total")
# print(f"  • SHAP analysis performed on {len(X_test_shap)} test samples")
# print("="*80)